# Single signal analysis — Moment scaling and autocorrelation

This notebook performs the moment scaling analysis and displacement autocorrelation analysis of individual seismic acceleration signals, following the framework of Vollmer et al. (2024). Results from the PDF 
analysis (notebook 03a) are imported at the end to build the complete summary table.

## 1. Imports and visualization settings

In [6]:
from pathlib import Path
import pandas as pd
from src.signals import (compute_moment_scaling_acc,compute_moment_scaling_vel, compute_moment_scaling_disp,
                         compute_scaling_exponents,test_scaling_linearity, fit_piecewise_scaling,
                         compute_displacement_autocorrelation, analyze_autocorrelation_scaling)
from src.plot_settings import set_plot_style
colors = set_plot_style()

## 2. Data loading

Two preprocessed datasets are loaded:
- `acc_preprocessed_all.parquet`: all 66 signals, used only for the autocorrelation analysis.
- `acc_preprocessed_long.parquet`: 48 signals with at least 48,000 samples, used for the moment scaling analysis (short-signal stations excluded).

In [7]:
df_acc_clean = pd.read_parquet('../data/processed/acc_preprocessed_all.parquet')
df_acc_long  = pd.read_parquet('../data/processed/acc_preprocessed_long.parquet')
print("All signals:", df_acc_clean['file'].nunique())
print("Long signals:", df_acc_long['file'].nunique())

All signals: 66
Long signals: 48


In [8]:
# Figures output directory
FIGURES_DIR = Path('../figures/03_single_signal')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

## 3. Moment scaling analysis

The scaling behavior of statistical moments is investigated to detect
signatures of anomalous or strongly anomalous diffusion, following the
framework of Vollmer et al. (2024).

For a process $x(t)$, the displacement over a time scale $\tau$ starting
from $t_0$ is defined as:

$$\Delta x(\tau, t_0) = x(t_0 + \tau) - x(t_0)$$

The $q$-th order moment is estimated as a temporal average over all
possible starting points $t_0$ (sliding window):

$$M_q(\tau) = \langle |\Delta x(\tau, t_0)|^q \rangle_{t_0}
= \frac{1}{N - \tau} \sum_{t_0=0}^{N-\tau-1} |\Delta x(\tau, t_0)|^q$$

If the process exhibits scaling, the moments obey a power law in $\tau$:

$$M_q(\tau) \sim \tau^{\zeta(q)}$$

For normal diffusion, $\zeta(q) = q/2$ (linear in $q$). Strong anomalous
diffusion is characterized by a piecewise-linear spectrum with a crossover
at $q = \alpha$ (Vollmer et al., 2024, Eq.~1b):

$$\zeta(q) = \begin{cases} \xi\, q & q \leq \alpha \\ \zeta_0\, q - \alpha(\zeta_0 - \xi) & q > \alpha \end{cases}$$

Three definitions of the process $x(t)$ are explored and compared:

- **Section 4** — $x(t) = a(t)$: the acceleration signal itself; increments
  are differences of acceleration values $\Delta a(\tau, t_0) = a(t_0+\tau) - a(t_0)$.
- **Section 5** — $x(t) = v(t) = \sum_{k} a(k)\,\Delta t$: the ground velocity,
  obtained by integrating the acceleration once.
- **Section 6** — $x(t) = \int v(t)\,dt$: the ground displacement,
  obtained by integrating the acceleration twice.

All three versions use a sampling interval $\Delta t = 0.005$\,s (200\,Hz).
The 6 near-field stations with short recordings (SURF, BRZ, BHB, CRI, SLZ, SAV)
are excluded from all moment scaling analyses, leaving 48 signals.

### 3.1 Acceleration

#### 3.1.1 Computation of q-th order moments

The process is defined as $x(t) = a(t)$, the normalized acceleration signal.
Increments are computed as:

$$\Delta a(\tau, t_0) = a(t_0 + \tau) - a(t_0)$$

Moments $M_q(\tau)$ are computed for moment orders $q \in \{0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0\}$ and time scales $\tau \in \{10, 50, 100, 200, 500, 1000, 2000, 5000, 10000\}$ samples.

In [ ]:
q_values_acc = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0]
tau_values_acc = [10, 20, 35, 50, 75, 100, 150, 200, 300, 500, 
              750, 1000, 1500, 2000, 3000, 5000, 7500, 10000]

In [10]:
df_moments_acc = compute_moment_scaling_acc(df_acc_long, q_values_acc, tau_values_acc)
print(df_moments_acc.shape)
display(df_moments_acc)

(3456, 6)


,file,station,stream,q,tau,moment
0,FR.EILF.00.HNE.D.INT-41004391.ACC.MP.ASC,EILF,HNE,0.5,10,0.591691
1,FR.EILF.00.HNE.D.INT-41004391.ACC.MP.ASC,EILF,HNE,1.0,10,0.587259
2,FR.EILF.00.HNE.D.INT-41004391.ACC.MP.ASC,EILF,HNE,1.5,10,0.938694
3,FR.EILF.00.HNE.D.INT-41004391.ACC.MP.ASC,EILF,HNE,2.0,10,2.129010
4,FR.EILF.00.HNE.D.INT-41004391.ACC.MP.ASC,EILF,HNE,2.5,10,5.949601
...,...,...,...,...,...,...
3451,RA.MENA.00.HNZ.D.INT-41004391.ACC.MP.ASC,MENA,HNZ,2.0,10000,2.524536
3452,RA.MENA.00.HNZ.D.INT-41004391.ACC.MP.ASC,MENA,HNZ,2.5,10000,5.608439
3453,RA.MENA.00.HNZ.D.INT-41004391.ACC.MP.ASC,MENA,HNZ,3.0,10000,13.615232
3454,RA.MENA.00.HNZ.D.INT-41004391.ACC.MP.ASC,MENA,HNZ,3.5,10000,35.341492


#### 3.1.2 Scaling exponents estimation

For each signal and each moment order $q$, the scaling exponent $\zeta(q)$ 
is estimated by linear regression of $\log M_q(\tau)$ vs $\log \tau$:

$$\log M_q(\tau) = \zeta(q) \log \tau + c$$

The slope $\zeta(q)$ is the scaling exponent. A plot of $\zeta(q)$ vs $q$ 
is produced for each signal, with the reference line $\zeta(q) = q/2$ 
corresponding to normal diffusion.

In [11]:
df_exponents_acc = compute_scaling_exponents(df_moments_acc, output_dir=FIGURES_DIR / 'scaling'/ 'acceleration' / 'exponents')

Saved: 48/48 individual plots
All individual plots saved successfully!


In [12]:
# Save results
try:
    df_exponents_acc.to_parquet('../data/processed/scaling_exponents_acc.parquet', index=False)
    print("Saved successfully!")
except Exception as e:
    print(f"Error saving file: {e}")

Saved successfully!


#### 3.1.3 Linearity check

The linearity of $\zeta(q)$ vs $q$ is assessed by comparing two models 
using AIC:

- **Linear**: $\zeta(q) = aq + b$
- **Quadratic**: $\zeta(q) = aq^2 + bq + c$

If the quadratic model is preferred, the scaling is non-linear, 
indicating anomalous diffusion.

A piecewise linear fit is also performed to detect the presence of 
two distinct scaling regimes, consistent with the framework of strong 
anomalous diffusion (Vollmer et al., 2021):

$$\zeta(q) = \begin{cases} \phi q & q \leq q^* \\ \lambda q - q^*(\lambda - \phi) & q > q^* \end{cases}$$

where $q^*$ is the breakpoint and $\phi$, $\lambda$ are the slopes of 
the two regimes. The breakpoint $q^*$ is estimated by minimizing the 
total residual sum of squares.

In [13]:
# Linearity check
df_linearity_acc = test_scaling_linearity(df_exponents_acc, output_dir=FIGURES_DIR / 'scaling'/ 'acceleration' / 'linearity')

Saved: 48/48 individual plots
All individual plots saved successfully!

Best fit by AIC:
best_fit
quadratic    46
linear        2
Name: count, dtype: int64


In [14]:
# Save results
try:
    df_linearity_acc.to_parquet('../data/processed/scaling_linearity_acc.parquet', index=False)
    print("Saved successfully!")
except Exception as e:
    print(f"Error saving file: {e}")

Saved successfully!


In [15]:
# Piecewise linear fit
df_piecewise_acc = fit_piecewise_scaling(df_exponents_acc, output_dir=FIGURES_DIR / 'scaling'/ 'acceleration' / 'piecewise')

Saved: 48/48 individual plots
All individual plots saved successfully!


In [16]:
# Save results
try:
    df_piecewise_acc.to_parquet('../data/processed/scaling_piecewise_acc.parquet', index=False)
    print("Saved successfully!")
except Exception as e:
    print(f"Error saving file: {e}")

Saved successfully!


### 3.2 Velocity

#### 3.2.1 Computation of q-th order moments

The process is defined as the ground velocity, obtained by integrating
the acceleration once:

$$v(t) = \sum_{k=0}^{t} a(k)\,\Delta t$$

Increments are computed as:

$$\Delta v(\tau, t_0) = v(t_0 + \tau) - v(t_0)$$

Moments $M_q(\tau)$ are computed for the same $q$ and $\tau$ values as
in Section 3.1.

In [ ]:
q_values_vel = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0]
tau_values_vel = [10, 20, 35, 50, 75, 100, 150, 200, 300, 500, 
              750, 1000, 1500, 2000, 3000, 5000, 7500, 10000]

In [18]:
df_moments_vel = compute_moment_scaling_vel(df_acc_long, q_values_vel, tau_values_vel)
print(df_moments_vel.shape)
display(df_moments_vel)

(3456, 6)


,file,station,stream,q,tau,moment
0,FR.EILF.00.HNE.D.INT-41004391.ACC.MP.ASC,EILF,HNE,0.5,10,0.075614
1,FR.EILF.00.HNE.D.INT-41004391.ACC.MP.ASC,EILF,HNE,1.0,10,0.011713
2,FR.EILF.00.HNE.D.INT-41004391.ACC.MP.ASC,EILF,HNE,1.5,10,0.003076
3,FR.EILF.00.HNE.D.INT-41004391.ACC.MP.ASC,EILF,HNE,2.0,10,0.001103
4,FR.EILF.00.HNE.D.INT-41004391.ACC.MP.ASC,EILF,HNE,2.5,10,0.000480
...,...,...,...,...,...,...
3451,RA.MENA.00.HNZ.D.INT-41004391.ACC.MP.ASC,MENA,HNZ,2.0,10000,0.006666
3452,RA.MENA.00.HNZ.D.INT-41004391.ACC.MP.ASC,MENA,HNZ,2.5,10000,0.003654
3453,RA.MENA.00.HNZ.D.INT-41004391.ACC.MP.ASC,MENA,HNZ,3.0,10000,0.002297
3454,RA.MENA.00.HNZ.D.INT-41004391.ACC.MP.ASC,MENA,HNZ,3.5,10000,0.001593


#### 3.2.2 Scaling exponents estimation

For each signal and each moment order $q$, the scaling exponent $\zeta(q)$ 
is estimated by linear regression of $\log M_q(\tau)$ vs $\log \tau$:

$$\log M_q(\tau) = \zeta(q) \log \tau + c$$

The slope $\zeta(q)$ is the scaling exponent. A plot of $\zeta(q)$ vs $q$ 
is produced for each signal, with the reference line $\zeta(q) = q/2$ 
corresponding to normal diffusion.

In [19]:
df_exponents_vel = compute_scaling_exponents(df_moments_vel, output_dir=FIGURES_DIR / 'scaling'/ 'velocity' / 'exponents')

Saved: 48/48 individual plots
All individual plots saved successfully!


In [20]:
# Save results
try:
    df_exponents_vel.to_parquet('../data/processed/scaling_exponents_vel.parquet', index=False)
    print("Saved successfully!")
except Exception as e:
    print(f"Error saving file: {e}")

Saved successfully!


#### 3.2.3 Linearity check

The linearity of $\zeta(q)$ vs $q$ is assessed by comparing two models 
using AIC:

- **Linear**: $\zeta(q) = aq + b$
- **Quadratic**: $\zeta(q) = aq^2 + bq + c$

If the quadratic model is preferred, the scaling is non-linear, 
indicating anomalous diffusion.

A piecewise linear fit is also performed to detect the presence of 
two distinct scaling regimes, consistent with the framework of strong 
anomalous diffusion (Vollmer et al., 2021):

$$\zeta(q) = \begin{cases} \phi q & q \leq q^* \\ \lambda q - q^*(\lambda - \phi) & q > q^* \end{cases}$$

where $q^*$ is the breakpoint and $\phi$, $\lambda$ are the slopes of 
the two regimes. The breakpoint $q^*$ is estimated by minimizing the 
total residual sum of squares.

In [21]:
# Linearity check
df_linearity_vel = test_scaling_linearity(df_exponents_vel, output_dir=FIGURES_DIR / 'scaling'/ 'velocity' / 'linearity')

Saved: 48/48 individual plots
All individual plots saved successfully!

Best fit by AIC:
best_fit
quadratic    47
linear        1
Name: count, dtype: int64


In [22]:
# Save results
try:
    df_linearity_vel.to_parquet('../data/processed/scaling_linearity_vel.parquet', index=False)
    print("Saved successfully!")
except Exception as e:
    print(f"Error saving file: {e}")

Saved successfully!


In [23]:
# Piecewise linear fit
df_piecewise_vel = fit_piecewise_scaling(df_exponents_vel, output_dir=FIGURES_DIR / 'scaling'/ 'velocity' / 'piecewise')

Saved: 48/48 individual plots
All individual plots saved successfully!


In [24]:
# Save results
try:
    df_piecewise_vel.to_parquet('../data/processed/scaling_piecewise_vel.parquet', index=False)
    print("Saved successfully!")
except Exception as e:
    print(f"Error saving file: {e}")

Saved successfully!


### 3.3 Displacement

#### 3.3.1 Computation of q-th order moments

The process is defined as the ground displacement, obtained by integrating
the acceleration twice:

$$v(t) = \sum_{k=0}^{t} a(k)\,\Delta t, \qquad
x(t) = \sum_{k=0}^{t} v(k)\,\Delta t$$

Increments are computed as:

$$\Delta x(\tau, t_0) = x(t_0 + \tau) - x(t_0)$$

Moments $M_q(\tau)$ are computed for the same $q$ and $\tau$ values as
in Sections 3.1 and 3.2.

In [ ]:
q_values_disp = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0]
tau_values_disp = [10, 20, 35, 50, 75, 100, 150, 200, 300, 500, 
              750, 1000, 1500, 2000, 3000, 5000, 7500, 10000]

In [26]:
df_moments_disp = compute_moment_scaling_disp(df_acc_long, q_values_disp, tau_values_disp)
print(df_moments_disp.shape)
display(df_moments_disp)

(3456, 6)


,file,station,stream,q,tau,moment
0,FR.EILF.00.HNE.D.INT-41004391.ACC.MP.ASC,EILF,HNE,0.5,10,1.647991e-02
1,FR.EILF.00.HNE.D.INT-41004391.ACC.MP.ASC,EILF,HNE,1.0,10,5.616708e-04
2,FR.EILF.00.HNE.D.INT-41004391.ACC.MP.ASC,EILF,HNE,1.5,10,3.113940e-05
3,FR.EILF.00.HNE.D.INT-41004391.ACC.MP.ASC,EILF,HNE,2.0,10,2.305301e-06
4,FR.EILF.00.HNE.D.INT-41004391.ACC.MP.ASC,EILF,HNE,2.5,10,2.050691e-07
...,...,...,...,...,...,...
3451,RA.MENA.00.HNZ.D.INT-41004391.ACC.MP.ASC,MENA,HNZ,2.0,10000,5.404816e-05
3452,RA.MENA.00.HNZ.D.INT-41004391.ACC.MP.ASC,MENA,HNZ,2.5,10000,8.313803e-06
3453,RA.MENA.00.HNZ.D.INT-41004391.ACC.MP.ASC,MENA,HNZ,3.0,10000,1.481448e-06
3454,RA.MENA.00.HNZ.D.INT-41004391.ACC.MP.ASC,MENA,HNZ,3.5,10000,2.942947e-07


#### 3.3.2 Scaling exponents estimation

For each signal and each moment order $q$, the scaling exponent $\zeta(q)$ 
is estimated by linear regression of $\log M_q(\tau)$ vs $\log \tau$:

$$\log M_q(\tau) = \zeta(q) \log \tau + c$$

The slope $\zeta(q)$ is the scaling exponent. A plot of $\zeta(q)$ vs $q$ 
is produced for each signal, with the reference line $\zeta(q) = q/2$ 
corresponding to normal diffusion.

In [27]:
df_exponents_disp = compute_scaling_exponents(df_moments_disp, output_dir=FIGURES_DIR / 'scaling'/ 'displacement' / 'exponents')

Saved: 48/48 individual plots
All individual plots saved successfully!


In [28]:
# Save results
try:
    df_exponents_disp.to_parquet('../data/processed/scaling_exponents_disp.parquet', index=False)
    print("Saved successfully!")
except Exception as e:
    print(f"Error saving file: {e}")

Saved successfully!


#### 3.3.3 Linearity check

The linearity of $\zeta(q)$ vs $q$ is assessed by comparing two models 
using AIC:

- **Linear**: $\zeta(q) = aq + b$
- **Quadratic**: $\zeta(q) = aq^2 + bq + c$

If the quadratic model is preferred, the scaling is non-linear, 
indicating anomalous diffusion.

A piecewise linear fit is also performed to detect the presence of 
two distinct scaling regimes, consistent with the framework of strong 
anomalous diffusion (Vollmer et al., 2021):

$$\zeta(q) = \begin{cases} \phi q & q \leq q^* \\ \lambda q - q^*(\lambda - \phi) & q > q^* \end{cases}$$

where $q^*$ is the breakpoint and $\phi$, $\lambda$ are the slopes of 
the two regimes. The breakpoint $q^*$ is estimated by minimizing the 
total residual sum of squares.

In [29]:
# Linearity check
df_linearity_disp = test_scaling_linearity(df_exponents_disp, output_dir=FIGURES_DIR / 'scaling'/ 'displacement' / 'linearity')

Saved: 48/48 individual plots
All individual plots saved successfully!

Best fit by AIC:
best_fit
quadratic    48
Name: count, dtype: int64


In [30]:
# Save results
try:
    df_linearity_disp.to_parquet('../data/processed/scaling_linearity_disp.parquet', index=False)
    print("Saved successfully!")
except Exception as e:
    print(f"Error saving file: {e}")

Saved successfully!


In [31]:
# Piecewise linear fit
df_piecewise_disp = fit_piecewise_scaling(df_exponents_disp, output_dir=FIGURES_DIR / 'scaling'/ 'displacement' / 'piecewise')

Saved: 48/48 individual plots
All individual plots saved successfully!


In [32]:
# Save results
try:
    df_piecewise_disp.to_parquet('../data/processed/scaling_piecewise_disp.parquet', index=False)
    print("Saved successfully!")
except Exception as e:
    print(f"Error saving file: {e}")

Saved successfully!


## 4. Displacement autocorrelation functions

The displacement autocorrelation function $C(t_1, t_2)$ is computed for
each signal following the framework of Vollmer et al. (2021). It captures
how the displacement process at time $t_1$ is correlated with its value
at time $t_2$:

$$C(t_1, t_2) = \langle x(t_1)\, x(t_2) \rangle$$

The function is evaluated on a logarithmic grid of $(t_1, t_2)$ pairs to
capture scaling behavior across multiple time scales. The scaling exponent
$\beta$ is estimated from the diagonal $C(t, t) \sim t^\beta$.

Three definitions of the process $x(t)$ are explored, consistent with
the choices made in Section 3.

### 4.1 Acceleration

The process is defined as $x(t) = a(t)$, the normalized acceleration signal.

In [33]:
df_autocorr_acc, C_matrices_acc = compute_displacement_autocorrelation(
    df_acc_long,
    output_dir=FIGURES_DIR / 'autocorrelation' / 'acceleration'
)

Saved: 48/48 individual plots
All individual plots saved successfully!


### 4.1.2 Scaling behavior

The scaling exponent $\beta$ is estimated from the diagonal $C(t, t)$,
which is expected to scale as:

$$C(t, t) = \langle x(t)^2 \rangle \sim t^\beta$$

The exponent $\beta$ is estimated by linear regression of
$\log C(t, t)$ vs $\log t$. A value of $\beta = 1$ corresponds to
normal diffusion ($\eta = 1$), while $\beta \neq 1$ indicates anomalous
scaling. In the framework of Vollmer et al. (2021), $\beta$ is directly
related to the mean-square displacement exponent $\eta$.

In [34]:
df_autocorr_scaling_acc = analyze_autocorrelation_scaling(
    C_matrices_acc,
    output_dir=FIGURES_DIR / 'autocorrelation' / 'acceleration'
)

Saved: 48/48 individual plots
All individual plots saved successfully!


In [35]:
try:
    df_autocorr_scaling_acc.to_parquet(
        '../data/processed/autocorr_scaling_acc.parquet', index=False)
    print("Saved successfully!")
except Exception as e:
    print(f"Error saving file: {e}")

Saved successfully!


### 4.2 Velocity

The process is defined as the ground velocity:
$$v(t) = \sum_{k=0}^{t} a(k)\,\Delta t$$

In [36]:
df_autocorr_vel, C_matrices_vel = compute_displacement_autocorrelation(
    df_acc_long,
    output_dir=FIGURES_DIR / 'autocorrelation' / 'velocity'
)

Saved: 48/48 individual plots
All individual plots saved successfully!


In [37]:
df_autocorr_scaling_vel = analyze_autocorrelation_scaling(
    C_matrices_vel,
    output_dir=FIGURES_DIR / 'autocorrelation' / 'velocity'
)

Saved: 48/48 individual plots
All individual plots saved successfully!


In [38]:
try:
    df_autocorr_scaling_vel.to_parquet(
        '../data/processed/autocorr_scaling_vel.parquet', index=False)
    print("Saved successfully!")
except Exception as e:
    print(f"Error saving file: {e}")

Saved successfully!


### 4.3 Displacement

The process is defined as the ground displacement:
$$v(t) = \sum_{k=0}^{t} a(k)\,\Delta t, \qquad x(t) = \sum_{k=0}^{t} v(k)\,\Delta t$$

In [39]:
df_autocorr_disp, C_matrices_disp = compute_displacement_autocorrelation(
    df_acc_long,
    output_dir=FIGURES_DIR / 'autocorrelation' / 'displacement'
)

Saved: 48/48 individual plots
All individual plots saved successfully!


In [40]:
df_autocorr_scaling_disp = analyze_autocorrelation_scaling(
    C_matrices_disp,
    output_dir=FIGURES_DIR / 'autocorrelation' / 'displacement'
)

Saved: 48/48 individual plots
All individual plots saved successfully!


In [41]:
try:
    df_autocorr_scaling_disp.to_parquet(
        '../data/processed/autocorr_scaling_disp.parquet', index=False)
    print("Saved successfully!")
except Exception as e:
    print(f"Error saving file: {e}")

Saved successfully!


## 5. Summary

A summary table collects the main results from all analyses for each signal.
For the moment scaling columns, results from the displacement version (Section 3.3)
are reported as the physically most meaningful choice.

| Column | Description |
|--------|-------------|
| `kurtosis` | Excess kurtosis $\kappa$ |
| `skewness` | Skewness $\gamma$ |
| `non_gaussian` | Anderson-Darling test result ($\alpha = 0.05$) |
| `best_fit_aic` | Best fitting distribution by AIC |
| `student_t_df` | Student-$t$ degrees of freedom $\nu$ |
| `power_law_exp` | Power law exponent $\hat{\alpha}$ (Hill estimator) |
| `q_break` | Piecewise scaling breakpoint $q^*$ |
| `slope_low` | Scaling slope for $q \leq q^*$ |
| `slope_high` | Scaling slope for $q > q^*$ |
| `beta` | Autocorrelation scaling exponent $\beta$ |


### 5.1 Acceleration

In [42]:
df_gaussian_results  = pd.read_parquet('../data/processed/gaussian_fit_results.parquet')
df_heavy_tail_results = pd.read_parquet('../data/processed/heavy_tail_results.parquet')
df_piecewise         = pd.read_parquet('../data/processed/scaling_piecewise_acc.parquet')
df_autocorr_scaling  = pd.read_parquet('../data/processed/autocorr_scaling.parquet')

df_summary = df_gaussian_results[['station', 'stream', 'kurtosis', 'skewness', 'non_gaussian']]\
    .merge(df_heavy_tail_results[['station', 'stream', 'best_fit_aic', 'student_t_df', 'power_law_exp']],
           on=['station', 'stream'])\
    .merge(df_piecewise[['station', 'stream', 'q_break', 'slope_low', 'slope_high']],
           on=['station', 'stream'])\
    .merge(df_autocorr_scaling[['station', 'stream', 'beta', 'r2']],
           on=['station', 'stream'])

display(df_summary)

try:
    df_summary.to_parquet('../data/processed/summary.parquet', index=False)
    print("Saved successfully!")
except Exception as e:
    print(f"Error saving file: {e}")

FileNotFoundError: [Errno 2] No such file or directory: '../data/processed/autocorr_scaling.parquet'

### 5.2 Velocity

### 5.3 Displacement